### Задача: Сегментация клиентов интернет-магазина. 
- Мы разделим покупателей на группы по их покупательской активности, чтобы маркетинговый отдел мог сделать им точечные предложения.

Мы будем использовать метод **RFM-анализа**:

- **Recency (Давность)** — как давно была последняя покупка.
- **Frequency (Частота)** — как часто клиент покупает.
- **Monetary (Деньги)** — сколько всего денег он потратил.

Решать задачу будем на **Python** с помощью алгоритма **K-Means**.

Шаг 1: Генерация синтетических данных

In [ ]:
import pandas as pd
import numpy as np

# Фиксируем генератор случайных чисел для повторяемости результатов
np.random.seed(42)

# Создаем данные для 1000 клиентов
n_customers = 1000

data = {
    'CustomerID': range(1, n_customers + 1),
    'Recency': np.random.randint(1, 365, size=n_customers),      # дней с последней покупки
    'Frequency': np.random.randint(1, 50, size=n_customers),     # количество покупок
    'Monetary': np.random.randint(100, 50000, size=n_customers)   # общая сумма в рублях
}

df = pd.DataFrame(data)
print(df.head())

Шаг 2: Масштабирование признаков (Важнейший шаг!)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Выбираем колонки для кластеризации
features = ['Recency', 'Frequency', 'Monetary']
X = df[features]

# Масштабируем данные
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Превращаем обратно в DataFrame для удобства
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

Шаг 3: Определение оптимального числа кластеров (Метод локтя)   

Мы не знаем заранее, на сколько групп делить клиентов. Запустим   
алгоритм несколько раз и посмотрим, как падает внутрикластерная  
 сумма расстояний (Inertia).

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

# Строим график метода локтя
plt.figure(figsize=(8, 4))
plt.plot(K_range, inertia, marker='o', linestyle='--')
plt.xlabel('Количество кластеров (K)')
plt.ylabel('Инерция (Inertia)')
plt.title('Метод локтя для поиска оптимального K')
plt.grid(True)
plt.show()

Шаг 4: Обучение модели K-Means

Возьмем, например, **3 кластера** и присвоим каждому клиенту его группу.

In [ ]:
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(df.head())

Шаг 5: Анализ и интерпретация результатов (Инсайт для бизнеса)

Самый главный шаг. Посчитаем средние значения признаков для каждого   
кластера, чтобы понять, кто в них находится.

In [ ]:
cluster_analysis = df.groupby('Cluster')[features].mean().round(1)
print("\nСредние показатели по кластерам:")
print(cluster_analysis)

---

- **Кластер 0 ("Спящие")**: Высокий `Recency` (давно не покупали), низкий `Frequency`.  

    _Решение маркетологов: отправить пуш-уведомление с большой скидкой, чтобы вернуть их._

- **Кластер 1 ("Постоянные/VIP")**: Низкий `Recency` (покупали недавно), высокий `Frequency` и `Monetary`.   

    _Решение маркетологов: предложить участие в закрытом клубе лояльности, скидки давать не нужно, они и так покупают._

- **Кластер 2 ("Новички")**: Низкий `Recency` (купили только что), но низкие `Frequency` и `Monetary`.  

    _Решение маркетологов: отправить приветственную цепочку писем с подборкой популярных товаров._

Шаг 6: Визуализация кластеров K-Means в 3D

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Каждому кластеру задаем свой цвет
colors = ['red', 'blue', 'green']

for cluster_id in range(optimal_k):
    cluster_data = df[df['Cluster'] == cluster_id]
    ax.scatter(
        cluster_data['Recency'], 
        cluster_data['Frequency'], 
        cluster_data['Monetary'], 
        c=colors[cluster_id], 
        label=f'Кластер {cluster_id}', 
        s=30,  # размер точек
        alpha=0.6
    )

ax.set_xlabel('Давность (Recency, дней)')
ax.set_ylabel('Частота (Frequency, покупок)')
ax.set_zlabel('Деньги (Monetary, руб)')
ax.set_title('3D-визуализация кластеров K-Means')
ax.legend()
plt.show()

Шаг 7: Пробуем алгоритм DBSCAN

Главное отличие DBSCAN — ему не нужно говорить «разбей на 3 части».   
Он сам поймет, где плотные скопления точек, а где — одиночные выбросы.

Для него важны два параметра:
- eps (Epsilon) — максимальный радиус вокруг точки, в котором мы ищем соседей.  

- min_samples — сколько минимум точек должно быть в этом радиусе, чтобы считать   
это место «плотным кластером».

Поскольку DBSCAN очень чувствителен к масштабу, мы обязательно скармливаем ему   
стандартизированные данные (X_scaled), которые подготовили на Шаге 2.

In [ ]:
from sklearn.cluster import DBSCAN

# Исправлено: имя переменной везде dbscan_tuned
dbscan_tuned = DBSCAN(eps=0.15, min_samples=10)
df['DBSCAN_Cluster_Tuned'] = dbscan_tuned.fit_predict(X_scaled)

print("Новое распределение клиентов по группам:")
print(df['DBSCAN_Cluster_Tuned'].value_counts())




Важнейший вывод этого кейса

Этот эксперимент наглядно доказал фундаментальное правило машинного обучения: **Нет универсального алгоритма под любую задачу.**

- `DBSCAN` идеален, когда в данных есть четкие геометрические пустоты и явные сгущения (например, треки на карте, скопления звезд, раздельные группы пикселей).   
На однородном «облаке» случайных данных он полностью ломается.  

- `K-Means` в этой бизнес-задаче победил безоговорочно. Даже на сплошном массиве данных он аккуратно «нарезал» пространство на логичные зоны, выделив VIP-клиентов,  
 новичков и уходящих покупателей.


Шаг 10: Финал. Выгрузка результатов для бизнеса

Так как `K-Means` справился отлично - сохраним результаты его работы.  
Добавим понятные текстовые названия для наших кластеров (на основе Шага 5)  
и выгрузим красивый CSV-файл, который маркетологи смогут загрузить в свою  
CRM-систему для рассылок.

In [ ]:
# Создаем словарь для переименования кластеров
cluster_names = {
    0: 'Спящие (Вернуть скидкой)',
    1: 'Постоянные VIP (Программа лояльности)',
    2: 'Новички (Приветственная рассылка)'
}

# Добавляем колонку с понятным описанием
df['Segment_Name'] = df['Cluster'].map(cluster_names)

# Оставляем только нужные для маркетинга колонки
final_report = df[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Segment_Name']]

# Сохраняем в файл
final_report.to_csv('customer_segmentation_results.csv', index=False, encoding='utf-8-sig')

print("Файл 'customer_segmentation_results.csv' успешно создан и готов к отправке в отдел маркетинга!")
print("\nПример итоговых данных:")
print(final_report.head())

#   CustomerID  Recency  Frequency  Monetary    Segment_Name
# 0           1      103         44     45450   Спящие (Вернуть скидкой) 
# 1           2      349          2     12301   Постоянные VIP (Программа лояльности)  
# 2           3      271         13     29526   Новички (Приветственная рассылка) 
